# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 4 — CTR / Engagement Opportunity Scoring → a ranking / scoring task.**

The output is not a yes/no answer for one page. It's an **ordered review queue**: an editor has time for a limited number of pages per week, so what matters is which pages sit at the top. That makes this a **ranking (scoring) task**, built on a classification-style signal underneath:

- **Score:** how likely a visible page is to keep under-capturing clicks for its position.
- **Queue:** pages sorted by that score, weighted by how many clicks are at stake.
- **Action the output supports:** an editor opens the top pages and rewrites title / meta description, checks intent match and snippet structure, or marks the page *monitor* when the cause is outside the page (a SERP feature, a navigational query).

Not clustering, because I'm not looking for page *types*. Not plain regression on CTR, because the decision is about order, not an exact CTR value.

In [1]:
TASK = {
    "lane": "Lane 4: CTR / Engagement Opportunity Scoring",
    "task_type": "ranking / scoring (classification signal underneath)",
    "unit": "one visible page (content item)",
    "output": "ranked review queue with score, missing clicks at tier CTR, reason codes",
    "action": "editor reviews title/meta/snippet/intent match, or marks 'monitor'",
}
for k, v in TASK.items():
    print(f"{k:>10}: {v}")

      lane: Lane 4: CTR / Engagement Opportunity Scoring
 task_type: ranking / scoring (classification signal underneath)
      unit: one visible page (content item)
    output: ranked review queue with score, missing clicks at tier CTR, reason codes
    action: editor reviews title/meta/snippet/intent match, or marks 'monitor'


## 2. Target or proxy

**Target (observed, later window):** `stays_under` = 1 if the page's CTR in the **most recent 30 days** is below **half of its position tier's pooled CTR** for the same 30 days, else 0.

**Features come from the window before it** (days 31–60 back: `impressions_prev_30d`, `clicks_prev_30d`), plus slow-moving context (intent, content type, freshness, engagement).

Why this shape:
- The label is an **observed outcome in a later window**, not a product flag and not a rule applied to the same numbers the model sees. That follows the framing rule *"the target must be observed, not defined."*
- It answers the editor's real question: *is this a lasting gap worth an edit, or a one-month wobble?*

Policy choices, written down so they can be challenged:
- **"Below half of tier CTR"** = clearly under-capturing, not a small miss.
- **≥ 300 impressions in both windows**, so a CTR estimate isn't built on a handful of views.
- **Positions 1–20 only** (`top_3`, `page_1`, `striking`), because below that, clicks are too rare to compare.

**Known weakness I'm keeping visible:** in the starter CSV, `position_tier` comes from a **90-day** average position, which overlaps the target window. That's mild leakage. In the warehouse I'll rebuild position from the feature window only.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/content_refresh_anonymized.csv").exists())
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")

VISIBLE_TIERS = ["top_3", "page_1", "striking"]
MIN_IMPR = 300          # per 30-day window
GAP = 0.5               # "below half of tier CTR"

lane = df[df["position_tier"].isin(VISIBLE_TIERS)].copy()
for w in ["prev", "last"]:
    imp, clk = lane[f"impressions_{w}_30d"], lane[f"clicks_{w}_30d"]
    lane[f"ctr_{w}_30d"] = 100 * clk / imp.where(imp > 0)          # x100 scale, like the CSV
    tier = lane.groupby("position_tier")[[f"clicks_{w}_30d", f"impressions_{w}_30d"]].sum()
    tier_ctr = 100 * tier[f"clicks_{w}_30d"] / tier[f"impressions_{w}_30d"]
    lane[f"tier_ctr_{w}"] = lane["position_tier"].map(tier_ctr)

lane = lane[(lane["impressions_prev_30d"] >= MIN_IMPR) & (lane["impressions_last_30d"] >= MIN_IMPR)].copy()
lane["under_prev"] = (lane["ctr_prev_30d"] < GAP * lane["tier_ctr_prev"]).astype(int)   # feature-window signal
lane["stays_under"] = (lane["ctr_last_30d"] < GAP * lane["tier_ctr_last"]).astype(int)  # TARGET (later window)

print(f"Eligible pages: {len(lane):,} across {lane['client_id'].nunique()} clients")
print(f"Target base rate (stays_under = 1): {lane['stays_under'].mean():.1%}")
print("\nTarget column, sketched:")
print(lane["stays_under"].value_counts().rename(index={0: "0 = captures clicks normally", 1: "1 = still under-captures"}))

Eligible pages: 8,155 across 28 clients
Target base rate (stays_under = 1): 35.3%

Target column, sketched:
stays_under
0 = captures clicks normally    5280
1 = still under-captures        2875
Name: count, dtype: int64


## 3. Success metric

**Precision@50 on a client-grouped holdout.** Of the 50 pages the queue puts on top, how many really still under-capture in the later window?

- **Why @50:** it matches how the list is used. An editor or small team works through a few dozen pages in a review cycle. I'll report **Precision@20** next to it for a single editor's week.
- **Why client-grouped:** pages from the same client share templates and audiences. Testing on clients the model never saw is the honest check (`client_id` for the split only, never as a feature).
- **Secondary metric:** share of all *missing clicks at tier CTR* captured by the top 50, so a queue full of tiny pages can't look good.

**What number means "good":** the transparent rule (rank by last month's click gap) already reaches the precision@50 printed below. A model is worth shipping only if it **clearly beats that rule on held-out clients**, for example by cutting the false positives at the top. If it doesn't, the rule *is* the answer, and I'll say so.

In [3]:
def precision_at_k(frame, score_col, k):
    top = frame.sort_values(score_col, ascending=False).head(k)
    return top["stays_under"].mean()

# Transparent baseline: rank by clicks missing vs. tier CTR in the FEATURE window (prev 30 days)
lane["missing_clicks_prev"] = (lane["tier_ctr_prev"] / 100 * lane["impressions_prev_30d"] - lane["clicks_prev_30d"]).clip(lower=0)

print(f"Base rate (random queue would score): {lane['stays_under'].mean():.1%}")
for k in (20, 50):
    print(f"Rule baseline precision@{k}: {precision_at_k(lane, 'missing_clicks_prev', k):.1%}")
print("\n(In-sample sanity check on the starter slice; the real comparison uses a client-grouped holdout.)")

Base rate (random queue would score): 35.3%
Rule baseline precision@20: 80.0%
Rule baseline precision@50: 82.0%

(In-sample sanity check on the starter slice; the real comparison uses a client-grouped holdout.)


## 4. The unit of analysis, as a real dataframe

**One row = one visible page (content item)** with its two 30-day windows side by side: *prev* (features) and *last* (target). Shown without IDs: `content_id` / `client_id` are pseudonyms for grouping and splitting only, and the rate columns are on the ×100 scale (`0.35` = 0.35%).

In [4]:
SHOW = ["position_tier", "main_intent", "content_type",
        "impressions_prev_30d", "clicks_prev_30d", "ctr_prev_30d", "tier_ctr_prev", "under_prev",
        "impressions_last_30d", "clicks_last_30d", "ctr_last_30d", "stays_under"]
print(f"Shape of the lane slice: {lane.shape[0]:,} rows (pages) x {len(SHOW)} shown columns")
print(f"Unique pages = rows: {lane['content_id'].nunique() == len(lane)}")
lane[SHOW].sample(6, random_state=7).round(3).reset_index(drop=True)

Shape of the lane slice: 8,155 rows (pages) x 12 shown columns
Unique pages = rows: True


,position_tier,main_intent,content_type,impressions_prev_30d,clicks_prev_30d,ctr_prev_30d,tier_ctr_prev,under_prev,impressions_last_30d,clicks_last_30d,ctr_last_30d,stays_under
0,page_1,transactional,keyword article,1966,4,0.203,0.340,0,333,2,0.601,0
1,page_1,informational,keyword article,1605,15,0.935,0.340,0,2234,24,1.074,0
2,page_1,informational,keyword article,1629,0,0.000,0.340,1,691,3,0.434,0
3,striking,informational,keyword article,592,1,0.169,0.346,1,311,1,0.322,0
4,page_1,transactional,keyword article,12226,34,0.278,0.340,0,8852,33,0.373,0
5,page_1,transactional,keyword article,29783,353,1.185,0.340,0,26853,277,1.032,0


## 5. Why ML beats a fixed rule here (or has to prove it does)

The fixed rule is *"flag pages under half of tier CTR last month."* It is informative, and the numbers below show it. They also show where it breaks:

1. **Many flags don't last.** A large share of pages under-capturing in one month are back to normal the next. An editor working from the raw rule spends hours on pages that would have recovered anyway. Telling a *lasting* gap from a *wobble* depends on several signals at once (impression volume, where the page sits inside its tier, intent, freshness, engagement), and a single threshold can't weigh them all.
2. **Tiers are coarse.** Pooled CTR for `striking` (11–20) comes out close to, or even above, `page_1` (4–10) in these windows. A 90-day average position blends pages that moved, so a fixed tier rule mislabels some pages. A model can use position as a continuous signal.
3. **No single feature separates the outcome.** Persistence is about the same across intents, so there's no easy if-statement to add. That's the case where a model (logistic regression first, then trees) can earn its place, if it beats the rule on held-out clients.

**Honest position:** this is an ML **and** analysis problem. The rule is the baseline, ML has to beat it, and the output stays *decision support*. A high score means "review this first", not "a rewrite will fix it".

In [5]:
flagged = lane[lane["under_prev"] == 1]
not_flagged = lane[lane["under_prev"] == 0]
print(f"Pages flagged by the rule last month: {len(flagged):,} ({len(flagged)/len(lane):.1%})")
print(f"  still under-capturing this month: {flagged['stays_under'].mean():.1%}")
print(f"  recovered on their own:          {1 - flagged['stays_under'].mean():.1%}  <- editor hours a better ranking could save")
print(f"Pages NOT flagged that drop under this month: {not_flagged['stays_under'].mean():.1%}  <- gaps the rule misses")

print("\nPersistence by intent (flagged pages) - no single feature separates it:")
print(flagged.groupby("main_intent")["stays_under"].agg(rate="mean", pages="size")
      .query("pages >= 30").round(3))

tier_ctr_view = lane.groupby("position_tier")[["tier_ctr_prev", "tier_ctr_last"]].first().reindex(VISIBLE_TIERS).round(3)
print("\nPooled tier CTR (%) per window - coarse tiers blur position:")
print(tier_ctr_view)

Pages flagged by the rule last month: 3,344 (41.0%)
  still under-capturing this month: 61.0%
  recovered on their own:          39.0%  <- editor hours a better ranking could save
Pages NOT flagged that drop under this month: 17.4%  <- gaps the rule misses

Persistence by intent (flagged pages) - no single feature separates it:
                rate  pages
main_intent                
commercial     0.628    589
informational  0.610   1987
transactional  0.597    740

Pooled tier CTR (%) per window - coarse tiers blur position:
               tier_ctr_prev  tier_ctr_last
position_tier                              
top_3                  0.562          0.566
page_1                 0.340          0.362
striking               0.346          0.409


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.